# CatBoost RMSE Optimization Notebook
Goal: Achieve ~0.1 RMSE range using feature engineering + CatBoost


In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error


In [ ]:
# Load data
train = pd.read_csv('/mnt/data/train.csv')
test = pd.read_csv('/mnt/data/test.csv')

train.head()

In [ ]:
# Feature Engineering
def feature_engineering(df):
    df = df.copy()
    
    # Height in inches
    df['Height_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    
    # BMI
    df['BMI'] = 703 * df['Weight(lb)'] / (df['Height_Inches'] ** 2)
    
    # Interaction features
    df['BPM_per_min'] = df['BPM'] * df['Exercise_Duration']
    df['Temp_x_Duration'] = df['Body_Temperature(F)'] * df['Exercise_Duration']
    df['Age_x_BPM'] = df['Age'] * df['BPM']
    
    return df

train = feature_engineering(train)
test = feature_engineering(test)

In [ ]:
# Target
target = 'Calories_Burned'
y = train[target]

# Drop ID and target
drop_cols = ['ID', target]
X = train.drop(columns=drop_cols)
X_test = test.drop(columns=['ID'])

In [ ]:
# Categorical columns
cat_cols = ['Gender', 'Weight_Status']

for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')

In [ ]:
# Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X))
preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
    val_pool = Pool(X_val, y_val, cat_features=cat_cols)
    
    model = CatBoostRegressor(
        iterations=5000,
        depth=6,
        learning_rate=0.03,
        loss_function='RMSE',
        eval_metric='RMSE',
        random_seed=42,
        early_stopping_rounds=200,
        verbose=200
    )
    
    model.fit(train_pool, eval_set=val_pool)
    
    oof[val_idx] = model.predict(X_val)
    preds += model.predict(X_test) / kf.n_splits
    
    rmse = mean_squared_error(y_val, oof[val_idx], squared=False)
    print(f'Fold {fold} RMSE:', rmse)

overall_rmse = mean_squared_error(y, oof, squared=False)
print('Overall RMSE:', overall_rmse)

In [ ]:
# Submission
submission = pd.DataFrame({
    'ID': test['ID'],
    'Calories_Burned': preds
})

submission.to_csv('/mnt/data/submission.csv', index=False)
submission.head()